In [4]:
import pandas as pd

In [8]:
zip_path = "/content/online+retail+ii(2).zip"

In [13]:
import zipfile

zip_path = "/content/online+retail+ii.zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("/content/dataset")

print("Dataset Extracted Successfully")

Dataset Extracted Successfully


In [5]:
import os

In [6]:
os.listdir('/content')

['.config', 'sample_data']

In [12]:
import os

os.listdir('/content')

['.config', 'online+retail+ii.zip', 'sample_data']

In [14]:
os.listdir('/content/dataset')

['online_retail_II.xlsx']

In [15]:
import pandas as pd

df = pd.read_excel('/content/dataset/online_retail_II.xlsx')

In [16]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [18]:
# Remove null values
df.dropna(inplace=True)

# Remove duplicates
df.drop_duplicates(inplace=True)

# Remove negative quantity
df = df[df['Quantity'] > 0]

# Remove negative price
df = df[df['Price'] > 0]

# Convert date
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# Create TotalPrice
df['TotalPrice'] = df['Quantity'] * df['Price']

# **CREATE RFM DATASET IN COLAB**

In [24]:
print(df.columns)

Index(['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'Price', 'Customer ID', 'Country', 'TotalPrice'],
      dtype='object')


In [25]:
# CREATE SNAPSHOT DATE
snapshot_date = df['InvoiceDate'].max()

# CREATE RFM TABLE
rfm = df.groupby('Customer ID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'Invoice': 'count',
    'TotalPrice': 'sum'
})

# RENAME COLUMNS
rfm.columns = ['Recency', 'Frequency', 'Monetary']

# RESET INDEX
rfm = rfm.reset_index()

rfm.head()

,Customer ID,Recency,Frequency,Monetary
0,12346.0,164,33,372.86
1,12347.0,2,71,1323.32
2,12348.0,73,20,222.16
3,12349.0,42,102,2671.14
4,12351.0,10,21,300.93


# **KMEANS**

In [26]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# SCALE DATA
scaler = StandardScaler()

rfm_scaled = scaler.fit_transform(
    rfm[['Recency', 'Frequency', 'Monetary']]
)

# APPLY KMEANS
kmeans = KMeans(
    n_clusters=4,
    random_state=42
)

rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)

rfm.head()

,Customer ID,Recency,Frequency,Monetary,Cluster
0,12346.0,164,33,372.86,1
1,12347.0,2,71,1323.32,0
2,12348.0,73,20,222.16,0
3,12349.0,42,102,2671.14,0
4,12351.0,10,21,300.93,0


In [27]:
rfm.to_csv("rfm_data.csv", index=False)

In [28]:
from google.colab import files

files.download("rfm_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [29]:
sales_forecast = df.groupby('InvoiceDate')['TotalPrice'].sum().reset_index()

sales_forecast.columns = ['Date', 'Sales']

sales_forecast.head()

,Date,Sales
0,2009-12-01 07:45:00,505.30
1,2009-12-01 07:46:00,145.80
2,2009-12-01 09:06:00,630.33
3,2009-12-01 09:08:00,310.75
4,2009-12-01 09:24:00,2286.24


In [30]:
sales_forecast.tail()

,Date,Sales
18003,2010-12-09 18:58:00,298.95
18004,2010-12-09 19:23:00,310.45
18005,2010-12-09 19:28:00,93.45
18006,2010-12-09 19:32:00,317.59
18007,2010-12-09 20:01:00,300.64


In [31]:
# Convert to date only
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

df['DateOnly'] = df['InvoiceDate'].dt.date

# Aggregate daily sales
sales_forecast = df.groupby('DateOnly')['TotalPrice'].sum().reset_index()

# Rename columns
sales_forecast.columns = ['Date', 'Sales']

sales_forecast.head()

,Date,Sales
0,2009-12-01,43894.87
1,2009-12-02,52762.06
2,2009-12-03,67413.62
3,2009-12-04,33913.81
4,2009-12-05,9803.05


In [32]:
sales_forecast.head()

,Date,Sales
0,2009-12-01,43894.87
1,2009-12-02,52762.06
2,2009-12-03,67413.62
3,2009-12-04,33913.81
4,2009-12-05,9803.05


In [33]:
sales_forecast.to_csv("forecast_data.csv", index=False)

In [34]:
from google.colab import files

files.download("forecast_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>